[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/31_flash_attention_solution.ipynb)

# 🔴 Solution: FlashAttention (tiled online softmax)

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `31_flash_attention.ipynb` first.

---
Implement attention the **FlashAttention** way: stream over key/value tiles,
never materializing the full $T_q \times T_k$ score matrix, and produce output
that is *numerically identical* to the standard implementation.

### Signature
```python
def flash_attention(q, k, v, block_size=16):
    # q: (T_q, d), k: (T_k, d), v: (T_k, d_v)
    ...  # -> (T_q, d_v)
```

### The online softmax recurrence
For each key block, with scores $s$ and current state $(m, \ell, \text{acc})$:

$$m^{\text{new}} = \max(m, \max s) \qquad
\alpha = e^{\,m - m^{\text{new}}}$$

$$\ell^{\text{new}} = \alpha\ell + \sum e^{\,s - m^{\text{new}}} \qquad
\text{acc}^{\text{new}} = \alpha\,\text{acc} + e^{\,s - m^{\text{new}}}V_{\text{block}}$$

Start at $m = -\infty$, $\ell = 0$, $\text{acc} = 0$; finish with
$\text{acc}/\ell$.

### Rules
- Process keys in tiles of `block_size`; never build the full score matrix
- Scale by $1/\sqrt{d}$
- Must be numerically exact against standard attention — not approximate
- Handle a `T_k` that is not a multiple of `block_size`
- Do not use `jax.nn.softmax` on the whole matrix

### Why the rescaling is what makes tiling *exact*
Softmax needs a global max for stability, but a streaming algorithm has not seen
the future when it processes block 1. The fix is to keep the max *so far*, and
when a later block raises it, retroactively correct everything already
accumulated by $e^{m_{\text{old}} - m_{\text{new}}}$.

Because $e^{s-m_{\text{old}}} \cdot e^{m_{\text{old}}-m_{\text{new}}} =
e^{s-m_{\text{new}}}$ exactly, the correction is not an approximation — it is an
algebraic identity. That is why FlashAttention is bit-comparable to the naive
version rather than an approximation like [[linear_attention]].

### It is an IO win, not a FLOP win
FlashAttention does the **same** number of floating-point operations as standard
attention — slightly more, in fact, because of the rescaling. It is faster
because attention is **memory-bandwidth bound**: the naive version writes an
$O(T^2)$ score matrix out to HBM and reads it back for the softmax and again for
the $V$ multiply. Flash keeps each tile in SRAM and never writes the scores at
all, cutting HBM traffic from $O(T^2)$ to $O(T^2/M)$ where $M$ is the SRAM tile
size.

The memory consequence is the bigger deal in practice: peak activation memory
for attention drops from $O(T^2)$ to $O(T)$, which is what made long context
affordable at all. Note that this Python/XLA version demonstrates the
*algorithm* — the actual speedup requires a fused kernel (Pallas/Triton/CUDA)
that controls SRAM residency directly.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def flash_attention(q, k, v, block_size=16):
    T_q, d = q.shape
    T_k, d_v = v.shape
    scale = 1.0 / jnp.sqrt(jnp.asarray(d, dtype=q.dtype))

    # Running state: max so far, denominator so far, unnormalised output so far.
    m = jnp.full((T_q, 1), -jnp.inf, dtype=q.dtype)
    ell = jnp.zeros((T_q, 1), dtype=q.dtype)
    acc = jnp.zeros((T_q, d_v), dtype=q.dtype)

    for start in range(0, T_k, block_size):
        stop = min(start + block_size, T_k)          # handles a ragged last tile
        k_blk = k[start:stop]
        v_blk = v[start:stop]

        s = (q @ k_blk.T) * scale                    # (T_q, blk)

        m_new = jnp.maximum(m, jnp.max(s, axis=-1, keepdims=True))
        # Retroactively correct everything accumulated under the OLD max.
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new)

        ell = alpha * ell + jnp.sum(p, axis=-1, keepdims=True)
        acc = alpha * acc + p @ v_blk
        m = m_new

    return acc / ell

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

q = jax.random.normal(jax.random.key(0), (8, 16))
k = jax.random.normal(jax.random.key(1), (40, 16))
v = jax.random.normal(jax.random.key(2), (40, 4))

ref = jax.nn.softmax(q @ k.T / jnp.sqrt(16.0), axis=-1) @ v

for bs in (4, 7, 16, 64):
    out = flash_attention(q, k, v, block_size=bs)
    print(f"block_size={bs:>3}: max |diff| vs standard = {float(jnp.abs(out - ref).max()):.2e}")
# Every tiling gives the same answer — the rescaling is exact, not approximate.

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("flash_attention")